# Syntactic predicate proposals

Inspect parser-proposed predicates in the existing 20-document smoke batch. These are not reference labels or B2 predictions. Task scope, semantic actor, attribution, polarity and presentation status remain unassigned. A known attachment failure is preserved in the tests and method note.


In [ ]:
from pathlib import Path
import json
import hashlib
import pandas as pd
from IPython.display import display, Markdown
current = Path.cwd()
ROOT = next((p for p in [current, *current.parents] if (p / "reports/syntactic_candidates_summary_v1.json").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Start from the project directory or notebooks folder.")
summary_path = ROOT / "data/processed/syntactic_candidates_summary_v1.json"
if not summary_path.exists():
    raise FileNotFoundError("Generate the local syntax output first; see archive/claim_syntax_experiments/SYNTACTIC_CANDIDATES_v1.md.")
summary = json.loads(summary_path.read_text())
local_output = ROOT / summary["local_output"]
if not local_output.exists():
    raise FileNotFoundError("Generate the local syntax output first; see archive/claim_syntax_experiments/SYNTACTIC_CANDIDATES_v1.md.")
payload = local_output.read_bytes()
if hashlib.sha256(payload).hexdigest() != summary["local_output_sha256"]:
    raise ValueError("Local parser output differs from the recorded run. Inspect versions before proceeding.")
parsed = json.loads(payload)
corpus_path = ROOT / "data/processed/model_corpus_v1.csv"
if hashlib.sha256(corpus_path.read_bytes()).hexdigest() != summary["provenance"]["corpus_sha256"]:
    raise ValueError("Frozen corpus differs.")
corpus = pd.read_csv(corpus_path, keep_default_na=False).set_index("execution_number")
display(pd.DataFrame(summary["document_counts"]))
print("Parser proposals only; more candidates do not establish higher recall.")


In [ ]:
def show_syntax(execution_number):
    document = next((d for d in parsed["documents"] if d["execution_number"] == execution_number), None)
    if document is None:
        raise ValueError("Choose an execution in the fixed 20-document batch.")
    display(Markdown(f"### Execution {execution_number}"))
    display(Markdown(corpus.loc[execution_number, "model_text"]))
    rows = []
    for item in document["predicate_candidates"]:
        rows.append({"focus": item["focus"]["text"], "start": item["focus"]["start"],
            "kind": item["kind"],
            "auxiliaries": ", ".join(t["text"] for t in item["auxiliaries"]),
            "subject_heads": ", ".join(t["text"] for t in item["subject_heads"]),
            "object_or_complement_heads": ", ".join(t["text"] for t in item["object_or_complement_heads"]),
            "governor": item["governing_predicate"]["text"] if item["governing_predicate"] else ""})
    display(pd.DataFrame(rows))
show_syntax(378)
